# 07 · 학습 — **Ablation 사다리**

**프로토콜 (전 그룹 공통)** — 학습 **150k step · seed 4개 · lr 고정(sweep 없음)**, eval **150k 체크포인트 × 5회 반복**(rep 마다 env seed 변경) → 모델당 4×5 = 20 run.

기여 3개(carry / BiMamba / MOSAIC)를 **각각 분리**. 없으면 기여 주장에 근거가 없음.

| 태그 | carry | BiMamba | MOSAIC | 어디서 |
|---|---|---|---|---|
| `acm` | ✗ | ✗ | ✗ | `03` |
| `acm_carry` | ✓ | ✗ | ✗ | **여기** |
| `acm_bimamba` | ✓ | ✓ | ✗ | **여기** |
| `acm_s7` | ✓ | ✗ | ✓ | **여기** (MOSAIC 을 BiMamba 없이 — 직교성) |
| **`ours`** | ✓ | ✓ | ✓ | `01` |

여기서 추가 학습 = 3모델 × 4 seed = **12잡** (`01`·`03` 을 먼저 돌린 전제).

**결정 실험 = `acm_bimamba` vs `ours`** — MOSAIC 이 SR 손해 없이 경계 떨림을 잡나.


In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import importlib, common_final as cf
importlib.reload(cf)

TASK  = cf.MAIN_SIM        # 'insertion' (aloha)
SEEDS = cf.MAIN_SEEDS      # [0,1,2,3] — 은지와 분담하면 여기만 바꿈 (예: [0,1])
GPUS  = cf.v23.available_gpus()   # 이 노드에 실제 보이는 GPU
NGPU  = len(GPUS)                 # 4개면 4잡씩 청크로 (하드코딩 X)
TAGS  = cf.GROUP_ABLATION  # ['acm_carry','acm_bimamba','acm_s7']

print('GPU  :', GPUS, f'({NGPU}개)')
print('학습:', TAGS, '| task:', TASK, '| seeds:', SEEDS)
print('steps:', f'{cf.STEPS:,}', '| 잡:', len(TAGS) * len(SEEDS))
for t in TAGS:
    pol, lr, K, extra, cp = cf.v23.MODEL_CONFIGS[t]
    print(f'  {t:<12} {pol:<34} lr={lr:<7} K={K:<4} pairs={cp}')

## 커맨드 확인 (dry-run)

In [ ]:
for t in TAGS:
    print(cf.make_train_cmd(t, seed=SEEDS[0], task=TASK, gpu_id=0))
    print()

## 학습 (resume 자동 — 끊겨도 다시 돌리면 이어감)
첫 실행은 **데이터셋을 한 번 먼저 내려받는다**(`cf.prefetch_dataset`). 이걸 건너뛰고 잡 N개를
동시에 띄우면 각자 같은 HF 캐시에 `snapshot_download` 를 호출해 서로의 미완성 파일을 읽고
`FileNotFoundError: ... meta/info.json` / `does not contain any parquet file` 로 죽는다
(다운로드를 완주한 잡 하나만 살아남음).

In [ ]:
jobs = cf.run_training(TAGS, SEEDS, task=TASK, ngpu=NGPU)

## 상태

In [ ]:
cf.print_training_status(jobs)
print()
cf.print_ckpt_status(TAGS, SEEDS, TASK)